In [1]:
!pip install -q openai pydantic pandas numpy scikit-learn joblib matplotlib

In [2]:
import os
from getpass import getpass

# 1. OpenAI API key (will prompt securely if not already set as a Colab secret)
if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

# 2. Experiment configuration
os.environ["RUN_API"] = "1"                      # "0" = dry run / prompt preview only, "1" = real API calls
os.environ["EXPERIMENT_TASK"] = "explanation"     # "explanation" (recommended) or "reconstruction"
os.environ["PILOT_N"] = "3"                       # number of applicants to test
os.environ["N_REPETITIONS"] = "1"                 # repeats per (applicant, condition) pair
os.environ["LLM_TEMPERATURE"] = "1"
os.environ["LLM_MODEL_ID"] = "gpt-5.6-sol"

Enter your OpenAI API key: ··········


In [3]:
"""LLM explanation benchmark for a frozen South German Credit model.

Recommended main experiment
---------------------------
    EXPERIMENT_TASK=explanation

The logistic-regression prediction is shown to the LLM and the experiment asks
which representation of the fitted model enables a faithful explanation.

Optional numerical control experiment
-------------------------------------
    EXPERIMENT_TASK=reconstruction

The prediction is hidden and the LLM is asked to reconstruct it when the
supplied information is sufficient.

The five prompt conditions are:
    1. customer_prediction_only
    2. textual_description
    3. training_code_only
    4. learned_parameters
    5. full_structured_package

Quick Colab setup:
    !pip install -q openai pydantic pandas numpy scikit-learn joblib

    import os
    os.environ["RUN_API"] = "0"  # prompt preview only
    os.environ["PILOT_N"] = "3"
    os.environ["EXPERIMENT_TASK"] = "explanation"
    %run south_german_credit_llm_experiment.py

For real API calls, change RUN_API to "1". The script securely asks for the
API key if OPENAI_API_KEY is not already set.
"""

from __future__ import annotations

import io
import json
import os
import time
import urllib.request
import zipfile
from getpass import getpass
from pathlib import Path
from typing import Literal

import joblib
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# ---------------------------------------------------------------------------
# 1. Experiment configuration
# ---------------------------------------------------------------------------

RANDOM_STATE = 42
TEST_SIZE = 0.30
CLASSIFICATION_THRESHOLD = 0.50
TOP_K = 5

CONDITIONS = [
    "customer_prediction_only",
    "textual_description",
    "training_code_only",
    "learned_parameters",
    "full_structured_package",
]

PILOT_N = int(os.getenv("PILOT_N", "3"))
N_REPETITIONS = int(os.getenv("N_REPETITIONS", "1"))
RUN_API = os.getenv("RUN_API", "0") == "1"
MODEL_ID = os.getenv("LLM_MODEL_ID", "gpt-5.6")
EXPERIMENT_TASK = os.getenv("EXPERIMENT_TASK", "explanation").strip().lower()
MAX_API_RETRIES = int(os.getenv("MAX_API_RETRIES", "3"))
TEMPERATURE = float(os.getenv("LLM_TEMPERATURE", "1"))

if EXPERIMENT_TASK not in {"explanation", "reconstruction"}:
    raise ValueError(
        "EXPERIMENT_TASK must be either 'explanation' or 'reconstruction'."
    )

RESULTS_DIR = Path(
    os.getenv("RESULTS_DIR", f"results_sgc_llm2/{EXPERIMENT_TASK}")
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# 2. Dataset dictionary
# ---------------------------------------------------------------------------

GERMAN_TO_ENGLISH = {
    "laufkont": "status",
    "laufzeit": "duration",
    "moral": "credit_history",
    "verw": "purpose",
    "hoehe": "amount",
    "sparkont": "savings",
    "beszeit": "employment_duration",
    "rate": "installment_rate",
    "famges": "personal_status_sex",
    "buerge": "other_debtors",
    "wohnzeit": "present_residence",
    "verm": "property",
    "alter": "age",
    "weitkred": "other_installment_plans",
    "wohn": "housing",
    "bishkred": "number_credits",
    "beruf": "job",
    "pers": "people_liable",
    "telef": "telephone",
    "gastarb": "foreign_worker",
    "kredit": "credit_risk",
}

NUMERIC_FEATURES = ["duration", "amount", "age"]

FEATURE_DESCRIPTIONS = {
    "status": "Status of the debtor's checking account with the bank.",
    "duration": "Credit duration in months.",
    "credit_history": "History of compliance with previous or concurrent credit contracts.",
    "purpose": "Purpose for which the credit is needed.",
    "amount": (
        "Credit amount in DM after an undocumented monotonic transformation; "
        "the original amount cannot be recovered."
    ),
    "savings": "Debtor's savings-account category.",
    "employment_duration": "Duration of employment with the current employer.",
    "installment_rate": "Installments as a percentage of disposable income.",
    "personal_status_sex": "Combined personal-status and sex category from the codebook.",
    "other_debtors": "Whether another debtor or guarantor supports the credit.",
    "present_residence": "Length of time at the present residence.",
    "property": "The debtor's most valuable property category.",
    "age": "Age in years.",
    "other_installment_plans": "Installment plans with providers other than this bank.",
    "housing": "Housing arrangement.",
    "number_credits": "Number of current or previous credits at this bank.",
    "job": "Employment or job qualification category.",
    "people_liable": "Number of people financially dependent on the debtor.",
    "telephone": "Whether a landline is registered in the customer's name.",
    "foreign_worker": "Whether the debtor is a foreign worker.",
}

CATEGORY_LABELS: dict[str, dict[int, str]] = {
    "status": {
        1: "no_checking_account",
        2: "checking_balance_below_0_DM",
        3: "checking_balance_0_to_below_200_DM",
        4: "checking_balance_at_least_200_DM_or_salary_account",
    },
    "credit_history": {
        0: "delay_in_paying_in_the_past",
        1: "critical_account_or_other_credits_elsewhere",
        2: "no_credits_taken_or_all_paid_back_duly",
        3: "existing_credits_paid_back_duly_until_now",
        4: "all_credits_at_this_bank_paid_back_duly",
    },
    "purpose": {
        0: "others",
        1: "new_car",
        2: "used_car",
        3: "furniture_or_equipment",
        4: "radio_or_television",
        5: "domestic_appliances",
        6: "repairs",
        7: "education",
        8: "vacation",
        9: "retraining",
        10: "business",
    },
    "savings": {
        1: "unknown_or_no_savings_account",
        2: "savings_below_100_DM",
        3: "savings_100_to_below_500_DM",
        4: "savings_500_to_below_1000_DM",
        5: "savings_at_least_1000_DM",
    },
    "employment_duration": {
        1: "unemployed",
        2: "employed_below_1_year",
        3: "employed_1_to_below_4_years",
        4: "employed_4_to_below_7_years",
        5: "employed_at_least_7_years",
    },
    "installment_rate": {
        1: "installment_rate_at_least_35_percent",
        2: "installment_rate_25_to_below_35_percent",
        3: "installment_rate_20_to_below_25_percent",
        4: "installment_rate_below_20_percent",
    },
    "personal_status_sex": {
        1: "male_divorced_or_separated",
        2: "female_non_single_or_male_single",
        3: "male_married_or_widowed",
        4: "female_single",
    },
    "other_debtors": {
        1: "none",
        2: "co_applicant",
        3: "guarantor",
    },
    "present_residence": {
        1: "residence_below_1_year",
        2: "residence_1_to_below_4_years",
        3: "residence_4_to_below_7_years",
        4: "residence_at_least_7_years",
    },
    "property": {
        1: "unknown_or_no_property",
        2: "car_or_other_property",
        3: "building_savings_agreement_or_life_insurance",
        4: "real_estate",
    },
    "other_installment_plans": {
        1: "bank",
        2: "stores",
        3: "none",
    },
    "housing": {
        1: "for_free",
        2: "rent",
        3: "own",
    },
    "number_credits": {
        1: "one_credit",
        2: "two_or_three_credits",
        3: "four_or_five_credits",
        4: "at_least_six_credits",
    },
    "job": {
        1: "unemployed_or_unskilled_non_resident",
        2: "unskilled_resident",
        3: "skilled_employee_or_official",
        4: "manager_self_employed_or_highly_qualified_employee",
    },
    "people_liable": {
        1: "three_or_more_people_liable",
        2: "zero_to_two_people_liable",
    },
    "telephone": {
        1: "no_telephone",
        2: "telephone_under_customer_name",
    },
    "foreign_worker": {
        1: "yes",
        2: "no",
    },
}


def normalize_category_code(value: object) -> int:
    """Convert values such as '2', 2, or 2.0 into the integer code 2."""

    return int(float(value))


def decode_raw_value(feature: str, value: object) -> str:
    """Return a readable raw applicant value using the supplied UCI codebook."""

    if feature in NUMERIC_FEATURES:
        return str(value)

    try:
        code = normalize_category_code(value)
    except (TypeError, ValueError):
        return str(value)

    label = CATEGORY_LABELS.get(feature, {}).get(code, f"unknown_code_{code}")
    return f"{code} ({label})"


def make_semantic_feature_name(transformed_name: str) -> str:
    """Convert sklearn names such as categorical__status_2 to readable names."""

    name = (
        transformed_name.replace("numeric__", "")
        .replace("categorical__", "")
    )

    if name in NUMERIC_FEATURES:
        return f"{name}_standardized"

    # Long variable names must be checked before a prefix can be misread.
    for variable in sorted(CATEGORY_LABELS, key=len, reverse=True):
        prefix = f"{variable}_"
        if name.startswith(prefix):
            code = normalize_category_code(name[len(prefix) :])
            label = CATEGORY_LABELS[variable].get(code, f"unknown_code_{code}")
            return f"{variable}={label}"

    return name


# ---------------------------------------------------------------------------
# 3. Dataset loading and frozen logistic-regression model
# ---------------------------------------------------------------------------

def load_south_german_credit() -> tuple[pd.DataFrame, pd.Series]:
    """Download corrected UCI dataset 573 and recode 0=bad to class 1."""

    url = (
        "https://archive.ics.uci.edu/static/public/573/"
        "south%2Bgerman%2Bcredit%2Bupdate.zip"
    )
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

    with urllib.request.urlopen(request, timeout=60) as response:
        archive_bytes = response.read()

    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        candidates = [
            name for name in archive.namelist()
            if name.lower().endswith("southgermancredit.asc")
        ]
        if not candidates:
            raise FileNotFoundError(
                "SouthGermanCredit.asc was not found inside the UCI ZIP file."
            )
        with archive.open(candidates[0]) as source:
            dataframe = pd.read_csv(source, sep=r"\s+")

    dataframe = dataframe.rename(columns=GERMAN_TO_ENGLISH)

    if "credit_risk" not in dataframe.columns:
        raise ValueError("Target column 'credit_risk' was not found.")

    X = dataframe.drop(columns=["credit_risk"]).copy()
    original_target = dataframe["credit_risk"].copy()

    # Original UCI coding: 0=bad, 1=good.
    # Experiment coding: 1=bad credit risk, 0=good credit risk.
    y_bad_credit = (original_target == 0).astype(int)
    y_bad_credit.name = "bad_credit_risk"

    if len(X) != 1000:
        raise ValueError(f"Expected 1000 rows but received {len(X)}.")
    if int(y_bad_credit.sum()) != 300:
        raise ValueError(
            f"Expected 300 bad-credit records but found {int(y_bad_credit.sum())}."
        )

    return X, y_bad_credit


def train_frozen_model(X: pd.DataFrame, y: pd.Series) -> dict:
    """Fit one reproducible pipeline and return all explanation artifacts."""

    X = X.copy()
    categorical_features = [
        column for column in X.columns if column not in NUMERIC_FEATURES
    ]

    for column in categorical_features:
        X[column] = X[column].astype(str)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), NUMERIC_FEATURES),
            (
                "categorical",
                OneHotEncoder(
                    drop="first",
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                categorical_features,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "logistic_regression",
                LogisticRegression(
                    max_iter=2000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )
    pipeline.fit(X_train, y_train)

    p_bad_credit = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (p_bad_credit >= CLASSIFICATION_THRESHOLD).astype(int)

    sklearn_names = (
        pipeline.named_steps["preprocessor"].get_feature_names_out().tolist()
    )
    semantic_names = [make_semantic_feature_name(name) for name in sklearn_names]

    if len(set(semantic_names)) != len(semantic_names):
        raise ValueError("Semantic feature names are not unique.")

    transformed = pipeline.named_steps["preprocessor"].transform(X_test)
    X_test_transformed = pd.DataFrame(
        transformed,
        columns=semantic_names,
        index=X_test.index,
    )

    logistic_model = pipeline.named_steps["logistic_regression"]
    coefficients = pd.Series(
        logistic_model.coef_[0],
        index=semantic_names,
        name="coefficient",
    )
    intercept = float(logistic_model.intercept_[0])

    fitted_preprocessor = pipeline.named_steps["preprocessor"]
    numeric_scaler = fitted_preprocessor.named_transformers_["numeric"]
    categorical_encoder = fitted_preprocessor.named_transformers_["categorical"]
    numeric_scaling = {
        feature: {
            "training_mean": float(mean),
            "training_standard_deviation": float(scale),
        }
        for feature, mean, scale in zip(
            NUMERIC_FEATURES,
            numeric_scaler.mean_,
            numeric_scaler.scale_,
        )
    }
    reference_categories = {
        feature: decode_raw_value(feature, categories[0])
        for feature, categories in zip(
            categorical_features,
            categorical_encoder.categories_,
        )
    }

    evaluation = pd.DataFrame(
        {
            "y_true": y_test,
            "p_bad_credit": p_bad_credit,
            "y_pred": y_pred,
        },
        index=X_test.index,
    )
    evaluation["outcome"] = np.select(
        [
            (evaluation.y_true == 1) & (evaluation.y_pred == 1),
            (evaluation.y_true == 0) & (evaluation.y_pred == 0),
            (evaluation.y_true == 0) & (evaluation.y_pred == 1),
            (evaluation.y_true == 1) & (evaluation.y_pred == 0),
        ],
        ["TP", "TN", "FP", "FN"],
        default="unknown",
    )

    performance = {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "roc_auc": float(roc_auc_score(y_test, p_bad_credit)),
        "pr_auc": float(average_precision_score(y_test, p_bad_credit)),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "n_transformed_features": int(len(semantic_names)),
        "positive_class": "1 = bad credit risk",
        "threshold": CLASSIFICATION_THRESHOLD,
    }

    return {
        "pipeline": pipeline,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "X_test_transformed": X_test_transformed,
        "coefficients": coefficients,
        "intercept": intercept,
        "evaluation": evaluation,
        "performance": performance,
        "categorical_features": categorical_features,
        "numeric_scaling": numeric_scaling,
        "reference_categories": reference_categories,
    }


def save_frozen_model_artifacts(artifacts: dict) -> None:
    """Save the exact pipeline, coefficients, and model metadata."""

    joblib.dump(artifacts["pipeline"], RESULTS_DIR / "frozen_pipeline.joblib")
    artifacts["coefficients"].rename_axis("feature").to_csv(
        RESULTS_DIR / "fitted_coefficients.csv"
    )
    (RESULTS_DIR / "model_performance.json").write_text(
        json.dumps(artifacts["performance"], indent=2),
        encoding="utf-8",
    )


# ---------------------------------------------------------------------------
# 4. Pilot cases and deterministic ground truth
# ---------------------------------------------------------------------------

def select_pilot_cases(evaluation: pd.DataFrame, n_cases: int) -> list[int]:
    """Select low-risk, high-risk, boundary, then outcome-diverse cases."""

    if n_cases < 1:
        raise ValueError("PILOT_N must be at least 1.")

    boundary_order = (
        (evaluation["p_bad_credit"] - CLASSIFICATION_THRESHOLD)
        .abs()
        .sort_values()
        .index.tolist()
    )

    if n_cases == 1:
        return [int(boundary_order[0])]

    selected: list[int] = [
        int(evaluation["p_bad_credit"].idxmin()),
        int(evaluation["p_bad_credit"].idxmax()),
    ]

    for case_id in boundary_order:
        if int(case_id) not in selected:
            selected.append(int(case_id))
            break

    # If more than three cases are requested, prefer outcome diversity.
    if n_cases > 3:
        for outcome in ["TP", "TN", "FP", "FN"]:
            candidates = evaluation.index[evaluation["outcome"] == outcome].tolist()
            for case_id in candidates:
                if int(case_id) not in selected:
                    selected.append(int(case_id))
                    break

    # Fill any remaining positions deterministically.
    for case_id in evaluation.sample(
        frac=1.0, random_state=RANDOM_STATE
    ).index.tolist():
        if int(case_id) not in selected:
            selected.append(int(case_id))
        if len(selected) >= n_cases:
            break

    return selected[: min(n_cases, len(evaluation))]


def make_case_artifact(case_id: int, artifacts: dict) -> dict:
    """Create the exact applicant-specific ground truth used for scoring."""

    raw_input = artifacts["X_test"].loc[case_id].to_dict()
    transformed_values = artifacts["X_test_transformed"].loc[case_id]
    coefficients = artifacts["coefficients"]
    contributions = coefficients * transformed_values.loc[coefficients.index]

    logit = float(artifacts["intercept"] + contributions.sum())
    probability = float(1.0 / (1.0 + np.exp(-logit)))
    predicted_class = int(probability >= CLASSIFICATION_THRESHOLD)

    sklearn_probability = float(
        artifacts["evaluation"].loc[case_id, "p_bad_credit"]
    )
    if not np.isclose(probability, sklearn_probability, atol=1e-10):
        raise AssertionError("Manual probability does not match sklearn output.")

    feature_table = pd.DataFrame(
        {
            "feature": coefficients.index,
            "coefficient": coefficients.values,
            "applicant_value": transformed_values.loc[coefficients.index].values,
            "true_contribution": contributions.loc[coefficients.index].values,
        }
    )
    feature_table["abs_contribution"] = feature_table[
        "true_contribution"
    ].abs()
    feature_table = feature_table.sort_values(
        ["abs_contribution", "feature"],
        ascending=[False, True],
    ).reset_index(drop=True)

    return {
        "case_id": int(case_id),
        "applicant_id": f"SGC{int(case_id) + 1:04d}",
        "raw_input": raw_input,
        "intercept": float(artifacts["intercept"]),
        "threshold": CLASSIFICATION_THRESHOLD,
        "feature_table": feature_table,
        "true_logit": logit,
        "true_probability": probability,
        "true_class": predicted_class,
        "true_classification": (
            "bad_credit" if predicted_class == 1 else "good_credit"
        ),
        "true_top_features": feature_table.head(TOP_K)["feature"].tolist(),
        "numeric_scaling": artifacts["numeric_scaling"],
        "reference_categories": artifacts["reference_categories"],
    }


def save_ground_truth_cases(cases: list[dict]) -> None:
    """Save one row per case and a long table of exact feature contributions."""

    case_rows = []
    contribution_rows = []

    for case in cases:
        case_rows.append(
            {
                "case_id": case["case_id"],
                "applicant_id": case["applicant_id"],
                "true_logit": case["true_logit"],
                "true_probability": case["true_probability"],
                "true_class": case["true_class"],
                "true_classification": case["true_classification"],
                "true_top_features": json.dumps(case["true_top_features"]),
            }
        )

        table = case["feature_table"].copy()
        table.insert(0, "applicant_id", case["applicant_id"])
        table.insert(0, "case_id", case["case_id"])
        contribution_rows.append(table)

    pd.DataFrame(case_rows).to_csv(
        RESULTS_DIR / "ground_truth_cases.csv", index=False
    )
    pd.concat(contribution_rows, ignore_index=True).to_csv(
        RESULTS_DIR / "ground_truth_contributions.csv", index=False
    )


# ---------------------------------------------------------------------------
# 5. Five controlled information conditions
# ---------------------------------------------------------------------------

def format_common_case(case: dict) -> str:
    """Information held constant across all prompt conditions."""

    lines = [
        "CASE",
        f"Applicant ID: {case['applicant_id']}",
        "Dataset target after recoding: 1 = bad credit risk; 0 = good credit risk",
        f"Classification threshold: {case['threshold']:.2f}",
        "",
        "RAW APPLICANT INPUT",
    ]

    for feature, value in case["raw_input"].items():
        lines.append(f"{feature}: {decode_raw_value(feature, value)}")

    if EXPERIMENT_TASK == "explanation":
        lines.extend(
            [
                "",
                "MODEL OUTPUT TO BE EXPLAINED",
                f"Bad-credit probability: {case['true_probability']:.10f}",
                f"Predicted class: {case['true_class']}",
                f"Classification label: {case['true_classification']}",
            ]
        )
    else:
        lines.extend(
            [
                "",
                "MODEL OUTPUT",
                "The model output is intentionally withheld in this reconstruction task.",
            ]
        )

    return "\n".join(lines)


def format_textual_description() -> str:
    """Algorithm and feature semantics; no fitted parameters are disclosed."""

    lines = ["MODEL AND FEATURE DESCRIPTION", """The prediction was produced by logistic regression. Logistic regression forms
a linear combination of transformed input features and learned coefficients,
then applies the sigmoid function to obtain the positive-class probability.
Positive applicant-specific model contributions increase the estimated
bad-credit risk and negative contributions decrease it.

Feature definitions:"""]
    for feature, description in FEATURE_DESCRIPTIONS.items():
        lines.append(f"- {feature}: {description}")
    lines.append("""
The fitted intercept, fitted coefficients, transformed applicant values, and
applicant-specific contributions are not supplied in this condition.""")
    return "\n".join(lines)


def format_training_code() -> str:
    """Actual training procedure, deliberately excluding learned parameters."""

    return f'''TRAINING CODE
```python
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = {NUMERIC_FEATURES!r}
categorical_features = [
    "status", "credit_history", "purpose", "savings",
    "employment_duration", "installment_rate", "personal_status_sex",
    "other_debtors", "present_residence", "property",
    "other_installment_plans", "housing", "number_credits", "job",
    "people_liable", "telephone", "foreign_worker",
]

# y_bad_credit was recoded so 1=bad credit and 0=good credit.
preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(
        drop="first", handle_unknown="ignore", sparse_output=False
    ), categorical_features),
])
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("logistic_regression", LogisticRegression(
        max_iter=2000, random_state={RANDOM_STATE}
    )),
])
pipeline.fit(X_train, y_train)
probability_bad_credit = pipeline.predict_proba(customer)[0, 1]
predicted_class = int(probability_bad_credit >= {CLASSIFICATION_THRESHOLD})
```

This condition shows how the model was trained, but it does not contain the
learned intercept, learned coefficients, or fitted scaling statistics.'''


def format_learned_parameters(case: dict) -> str:
    """Structured learned parameters and exact transformed customer vector."""

    rows = case["feature_table"].sort_values("feature")
    lines = [
        "LEARNED MODEL PARAMETERS",
        "Model type: logistic regression",
        "Positive class: 1 = bad credit risk",
        "Equation: z = intercept + sum(coefficient_j * transformed_value_j)",
        "Probability: sigmoid(z)",
        "Numeric preprocessing: (raw value - training mean) / training standard deviation",
        "Categorical preprocessing: one-hot encoding with the first category as reference.",
        f"Intercept: {case['intercept']:.10f}",
        "",
        "FITTED NUMERIC SCALING",
        "| feature | training_mean | training_standard_deviation |",
        "|---|---:|---:|",
    ]
    for feature, values in case["numeric_scaling"].items():
        lines.append(
            f"| {feature} | {values['training_mean']:.10f} | "
            f"{values['training_standard_deviation']:.10f} |"
        )

    lines.extend(["", "ONE-HOT REFERENCE CATEGORIES"])
    for feature, reference in case["reference_categories"].items():
        lines.append(f"- {feature}: {reference}")

    lines.extend([
        "",
        "| transformed_feature | fitted_coefficient | transformed_customer_value |",
        "|---|---:|---:|",
    ])
    for row in rows.itertuples():
        lines.append(
            f"| {row.feature} | {row.coefficient:.10f} | "
            f"{row.applicant_value:.10f} |"
        )
    lines.append(
        "\nApplicant-specific contributions are not precomputed in this condition."
    )
    return "\n".join(lines)


def format_full_structured_package(case: dict) -> str:
    """Complete machine-verifiable evidence including exact contributions."""

    rows = case["feature_table"].sort_values("feature")
    package = {
        "target": {
            "positive_class": "bad_credit",
            "positive_class_value": 1,
            "negative_class": "good_credit",
            "negative_class_value": 0,
            "decision_threshold": case["threshold"],
        },
        "formula": {
            "logit": "intercept + sum(log_odds_contribution)",
            "probability": "sigmoid(logit)",
            "feature_contribution": "coefficient * transformed_customer_value",
        },
        "intercept": case["intercept"],
        "numeric_scaling": case["numeric_scaling"],
        "one_hot_reference_categories": case["reference_categories"],
        "features": [
            {
                "feature": row.feature,
                "coefficient": round(float(row.coefficient), 10),
                "transformed_customer_value": round(float(row.applicant_value), 10),
                "log_odds_contribution": round(float(row.true_contribution), 10),
            }
            for row in rows.itertuples()
        ],
        "calculation_check": {
            "logit": case["true_logit"],
            "probability_bad_credit": case["true_probability"],
            "predicted_class": case["true_class"],
        },
    }
    return "COMPLETE STRUCTURED EVIDENCE PACKAGE\n```json\n" + json.dumps(
        package, indent=2
    ) + "\n```"


EXPLANATION_TASK = """TASK
Explain why the trained logistic-regression model produced this prediction for
this applicant. Base your explanation only on the supplied information and
explicitly identify any limitations."""


RECONSTRUCTION_TASK = """TASK
Reconstruct and explain the trained logistic-regression model's withheld
prediction for this applicant. Base your answer only on the supplied information
and explicitly identify any limitations."""


def build_prompts(case: dict) -> dict[str, str]:
    """Keep the case and task fixed; vary only the supplied evidence block."""

    common_case = format_common_case(case)
    textual_description = format_textual_description()
    training_code = format_training_code()
    learned_parameters = format_learned_parameters(case)
    full_package = format_full_structured_package(case)
    task = EXPLANATION_TASK if EXPERIMENT_TASK == "explanation" else RECONSTRUCTION_TASK

    evidence_blocks = {
        "customer_prediction_only": "",
        "textual_description": textual_description,
        "training_code_only": training_code,
        "learned_parameters": learned_parameters,
        "full_structured_package": full_package,
    }

    prompts: dict[str, str] = {}
    for condition, evidence in evidence_blocks.items():
        # The condition label is deliberately not shown to the LLM. It is kept
        # only as the external dictionary key used by the experiment runner.
        sections = [common_case]
        if evidence:
            sections.append(evidence)
        sections.append(task)
        prompts[condition] = "\n\n".join(sections)

    return prompts


# ---------------------------------------------------------------------------
# 6. Structured LLM output
# ---------------------------------------------------------------------------

Direction = Literal[
    "increase_bad_credit_risk",
    "decrease_bad_credit_risk",
    "neutral",
]

class FeatureAttribution(BaseModel):
    feature: str = Field(
        description="Feature name exactly as supplied in the prompt."
    )
    customer_value: str | None = Field(
        description=(
            "Human-readable raw customer value when it can be matched from "
            "the prompt; otherwise null."
        )
    )
    coefficient: float | None = Field(
        description="Fitted logistic-regression coefficient, or null."
    )
    transformed_value: float | None = Field(
        description="Exact transformed applicant value used by the model, or null."
    )
    contribution: float | None = Field(
        description="Numerical feature contribution when supported; otherwise null."
    )
    direction: Direction | None = Field(
        description="Direction of the applicant-specific model contribution."
    )
    explanation: str = Field(
        description=(
            "Short, non-causal explanation of how this supplied contribution "
            "moves the model's bad-credit log-odds."
        )
    )


class LLMAnswer(BaseModel):
    explanation_possible: bool = Field(
        description="Whether the supplied information supports the requested explanation."
    )
    reconstructed_logit: float | None = Field(
        description="Independently reconstructed logit, or null if unavailable."
    )
    reconstructed_probability: float | None = Field(
        description="Reconstructed positive-class probability, or null."
    )
    reconstructed_class: int | None = Field(
        description="Reconstructed class 0/1, or null if unavailable."
    )
    top_features: list[FeatureAttribution] = Field(
        description="Zero to five strongest supported feature contributions."
    )
    model_explanation: str = Field(
        description=(
            "A self-contained customer-level model explanation. With sufficient "
            "evidence it must name the strongest actual features, their directions, "
            "the logit, probability, threshold, and class. With insufficient "
            "evidence it must abstain from feature-level claims."
        )
    )
    limitations: list[str] = Field(
        description=(
            "Limitations or missing information identified from the supplied "
            "evidence; use an empty list only when none are identified."
        )
    )


# Resolve postponed annotations explicitly for notebook/import compatibility.
FeatureAttribution.model_rebuild()
LLMAnswer.model_rebuild()


SYSTEM_INSTRUCTION = f"""Analyze the supplied frozen logistic-regression model
and applicant information. Follow the user's task and return data matching the
supplied Pydantic schema.

Use only information in the user message. Do not introduce unsupported facts or
feature effects from general credit-risk knowledge. If a structured field cannot
be supported from the supplied information, use null, an empty list, or explain
the limitation as appropriate. Use supplied feature names exactly. Return no
more than {TOP_K} top_features. Distinguish model associations from causal claims
and do not present a prediction as a confirmed future outcome."""


def get_openai_client():
    """Create the client and request a hidden API key only when necessary."""

    from openai import OpenAI

    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ").strip()

    if not os.environ["OPENAI_API_KEY"]:
        raise ValueError("OPENAI_API_KEY is empty.")

    return OpenAI()


def call_llm(client, prompt: str) -> tuple[LLMAnswer, str, str, dict]:
    """Call the LLM with Structured Outputs and retry transient failures."""

    last_error: Exception | None = None

    for attempt in range(1, MAX_API_RETRIES + 1):
        try:
            completion = client.chat.completions.parse(
                model=MODEL_ID,
                temperature=TEMPERATURE,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": prompt},
                ],
                response_format=LLMAnswer,
            )

            choice = completion.choices[0]
            if choice.message.refusal:
                raise RuntimeError(f"Model refusal: {choice.message.refusal}")

            answer = choice.message.parsed
            if answer is None:
                raise RuntimeError("The API returned no parsed answer.")

            usage = {
                "prompt_tokens": getattr(completion.usage, "prompt_tokens", None),
                "completion_tokens": getattr(
                    completion.usage, "completion_tokens", None
                ),
                "total_tokens": getattr(completion.usage, "total_tokens", None),
            }

            return (
                answer,
                choice.message.content or answer.model_dump_json(),
                completion.id,
                usage,
            )
        except Exception as exc:  # Retry behavior is logged in the final result.
            last_error = exc
            if attempt == MAX_API_RETRIES:
                break
            time.sleep(min(2 ** (attempt - 1), 8))

    assert last_error is not None
    raise last_error


# ---------------------------------------------------------------------------
# 7. Deterministic scoring
# ---------------------------------------------------------------------------

def true_direction(contribution: float, tolerance: float = 1e-12) -> str:
    if contribution > tolerance:
        return "increase_bad_credit_risk"
    if contribution < -tolerance:
        return "decrease_bad_credit_risk"
    return "neutral"


def optional_absolute_error(predicted: float | None, truth: float) -> float:
    if predicted is None:
        return np.nan
    return abs(float(predicted) - float(truth))


def score_answer(answer: LLMAnswer, case: dict, condition: str) -> dict:
    """Compare structured claims with exact coefficient-times-value truth."""

    truth = case["feature_table"].set_index("feature")
    valid_features = set(truth.index)
    returned_features = [item.feature for item in answer.top_features]
    true_top = case["true_top_features"]

    has_exact_evidence = condition in {
        "learned_parameters",
        "full_structured_package",
    }
    expected_possible = has_exact_evidence

    limitation_text = " ".join(
        [*answer.limitations, answer.model_explanation]
    ).lower()
    missing_model_information_terms = {
        "coefficient",
        "intercept",
        "fitted parameter",
        "learned parameter",
        "transformed value",
        "transformed feature",
        "feature contribution",
        "scaling statistic",
        "model parameter",
    }
    discovered_missing_model_information = any(
        term in limitation_text for term in missing_model_information_terms
    )

    hallucinated = [
        feature for feature in returned_features if feature not in valid_features
    ]
    overlap_count = len(set(returned_features).intersection(true_top))

    direction_correct: list[int] = []
    contribution_errors: list[float] = []

    for item in answer.top_features:
        if item.feature not in valid_features:
            continue

        expected_contribution = float(
            truth.loc[item.feature, "true_contribution"]
        )

        if item.direction is not None:
            direction_correct.append(
                int(item.direction == true_direction(expected_contribution))
            )

        if item.contribution is not None:
            contribution_errors.append(
                abs(float(item.contribution) - expected_contribution)
            )

    # Feature fidelity is undefined for the deliberately insufficient baseline.
    top5_recall = overlap_count / TOP_K if has_exact_evidence else np.nan

    # When the true output is shown in explanation mode, numerical fields can
    # be copied and therefore are not valid reconstruction metrics.
    if EXPERIMENT_TASK == "reconstruction":
        logit_abs_error = optional_absolute_error(
            answer.reconstructed_logit, case["true_logit"]
        )
        probability_abs_error = optional_absolute_error(
            answer.reconstructed_probability, case["true_probability"]
        )
        reconstructed_class_error = np.nan
        if answer.reconstructed_class is not None:
            reconstructed_class_error = int(
                int(answer.reconstructed_class) != int(case["true_class"])
            )
    else:
        logit_abs_error = np.nan
        probability_abs_error = np.nan
        reconstructed_class_error = np.nan

    return {
        "explanation_possible_correct": int(
            answer.explanation_possible == expected_possible
        ),
        "appropriate_abstention": int(
            (not expected_possible and not answer.explanation_possible)
            or (expected_possible and answer.explanation_possible)
        ),
        "missing_model_information_identified": (
            int(discovered_missing_model_information)
            if not has_exact_evidence
            else np.nan
        ),
        "unsupported_feature_claim": int(
            not has_exact_evidence and len(returned_features) > 0
        ),
        "logit_abs_error": logit_abs_error,
        "probability_abs_error": probability_abs_error,
        "reconstructed_class_error": reconstructed_class_error,
        "top5_recall": top5_recall,
        "direction_accuracy": (
            float(np.mean(direction_correct)) if direction_correct else np.nan
        ),
        "contribution_mae": (
            float(np.mean(contribution_errors))
            if contribution_errors
            else np.nan
        ),
        "hallucination_rate": len(hallucinated)
        / max(1, len(returned_features)),
        "returned_feature_count": len(returned_features),
        "too_many_features": int(len(returned_features) > TOP_K),
        "hallucinated_features": json.dumps(hallucinated),
    }


# ---------------------------------------------------------------------------
# 8. Experiment runner, compact outputs, and summaries
# ---------------------------------------------------------------------------

def compact_answer_record(
    answer: LLMAnswer,
    case: dict,
    condition: str,
) -> dict:
    """Create a readable JSON record while retaining detailed benchmark output."""

    return {
        "applicant_id": case["applicant_id"],
        "classification": case["true_classification"],
        "bad_credit_probability": round(case["true_probability"], 6),
        "prompt_id": condition,
        "explanation_possible": answer.explanation_possible,
        "reconstructed_logit": answer.reconstructed_logit,
        "reconstructed_probability": answer.reconstructed_probability,
        "reconstructed_class": answer.reconstructed_class,
        "top_features": [
            feature.model_dump() for feature in answer.top_features
        ],
        "model_explanation": answer.model_explanation,
        "limitations": answer.limitations,
    }


def run_experiment(cases: list[dict]) -> pd.DataFrame:
    client = get_openai_client()
    results: list[dict] = []
    compact_records: list[dict] = []
    checkpoint = RESULTS_DIR / "api_results_checkpoint.csv"

    for repetition in range(1, N_REPETITIONS + 1):
        for case in cases:
            prompts = build_prompts(case)

            for condition, prompt in prompts.items():
                print(
                    f"repetition={repetition} "
                    f"applicant={case['applicant_id']} condition={condition}"
                )

                try:
                    answer, raw_answer, response_id, usage = call_llm(client, prompt)
                    scores = score_answer(answer, case, condition)
                    parsed_answer = answer.model_dump_json()
                    model_explanation = answer.model_explanation
                    explanation_possible = answer.explanation_possible
                    error_message = ""
                    compact_records.append(
                        compact_answer_record(answer, case, condition)
                    )
                except Exception as exc:
                    response_id = ""
                    raw_answer = ""
                    parsed_answer = ""
                    model_explanation = ""
                    explanation_possible = np.nan
                    usage = {
                        "prompt_tokens": None,
                        "completion_tokens": None,
                        "total_tokens": None,
                    }
                    scores = {}
                    error_message = f"{type(exc).__name__}: {exc}"

                results.append(
                    {
                        "experiment_task": EXPERIMENT_TASK,
                        "repetition": repetition,
                        "case_id": case["case_id"],
                        "applicant_id": case["applicant_id"],
                        "condition": condition,
                        "model_id": MODEL_ID,
                        "temperature": TEMPERATURE,
                        "response_id": response_id,
                        "prompt": prompt,
                        "prompt_character_count": len(prompt),
                        "prompt_word_count": len(prompt.split()),
                        "raw_answer": raw_answer,
                        "parsed_answer": parsed_answer,
                        "model_explanation": model_explanation,
                        "explanation_possible": explanation_possible,
                        "true_logit": case["true_logit"],
                        "true_probability": case["true_probability"],
                        "true_class": case["true_class"],
                        "true_classification": case["true_classification"],
                        "true_top_features": json.dumps(
                            case["true_top_features"]
                        ),
                        "prompt_tokens": usage["prompt_tokens"],
                        "completion_tokens": usage["completion_tokens"],
                        "total_tokens": usage["total_tokens"],
                        "error_message": error_message,
                        **scores,
                    }
                )

                pd.DataFrame(results).to_csv(checkpoint, index=False)
                time.sleep(0.2)

    with (RESULTS_DIR / "compact_llm_answers.jsonl").open(
        "w", encoding="utf-8"
    ) as output:
        for record in compact_records:
            output.write(json.dumps(record, ensure_ascii=False) + "\n")

    return pd.DataFrame(results)


def summarize_results(results: pd.DataFrame) -> pd.DataFrame:
    metric_columns = [
        "explanation_possible_correct",
        "appropriate_abstention",
        "missing_model_information_identified",
        "unsupported_feature_claim",
        "logit_abs_error",
        "probability_abs_error",
        "reconstructed_class_error",
        "top5_recall",
        "direction_accuracy",
        "contribution_mae",
        "hallucination_rate",
        "too_many_features",
        "prompt_character_count",
        "prompt_word_count",
        "prompt_tokens",
        "completion_tokens",
        "total_tokens",
    ]
    available = [column for column in metric_columns if column in results.columns]
    return results.groupby("condition", dropna=False)[available].mean()


def create_human_review_template(results: pd.DataFrame) -> pd.DataFrame:
    """Create an optional 0-2 manual rubric for the free-text explanations."""

    review = results[
        [
            "experiment_task",
            "repetition",
            "applicant_id",
            "condition",
            "model_explanation",
            "true_top_features",
        ]
    ].copy()
    review["human_faithfulness_0_to_2"] = ""
    review["human_clarity_0_to_2"] = ""
    review["human_no_unsupported_claims_0_to_2"] = ""
    review["reviewer_notes"] = ""
    return review


def save_prompt_previews(cases: list[dict]) -> None:
    preview_dir = RESULTS_DIR / "prompt_previews"
    preview_dir.mkdir(exist_ok=True)

    for case in cases:
        for condition, prompt in build_prompts(case).items():
            path = preview_dir / f"{case['applicant_id']}_{condition}.txt"
            path.write_text(prompt, encoding="utf-8")


def show(value) -> None:
    """Use notebook display when available, otherwise print."""

    try:
        from IPython.display import display

        display(value)
    except ImportError:
        print(value)


# ---------------------------------------------------------------------------
# 9. Main execution
# ---------------------------------------------------------------------------

def main() -> None:
    print("Loading South German Credit from UCI...")
    X, y_bad_credit = load_south_german_credit()
    print(f"Dataset shape: X={X.shape}, y={y_bad_credit.shape}")

    print("\nTraining one reproducible frozen logistic-regression model...")
    artifacts = train_frozen_model(X, y_bad_credit)
    save_frozen_model_artifacts(artifacts)
    print(json.dumps(artifacts["performance"], indent=2))

    case_ids = select_pilot_cases(artifacts["evaluation"], PILOT_N)
    cases = [make_case_artifact(case_id, artifacts) for case_id in case_ids]
    save_ground_truth_cases(cases)
    save_prompt_previews(cases)

    selection = artifacts["evaluation"].loc[case_ids].copy()
    selection.insert(
        0,
        "applicant_id",
        [f"SGC{int(case_id) + 1:04d}" for case_id in selection.index],
    )
    selection.index.name = "case_id"
    selection.to_csv(RESULTS_DIR / "selected_cases.csv")

    print("\nSelected pilot cases:")
    show(selection)

    design = pd.DataFrame(
        [
            {
                "condition": "customer_prediction_only",
                "customer_and_prediction": 1,
                "textual_description": 0,
                "training_code": 0,
                "learned_parameters": 0,
                "precomputed_contributions": 0,
                "exact_feature_explanation_possible": 0,
            },
            {
                "condition": "textual_description",
                "customer_and_prediction": 1,
                "textual_description": 1,
                "training_code": 0,
                "learned_parameters": 0,
                "precomputed_contributions": 0,
                "exact_feature_explanation_possible": 0,
            },
            {
                "condition": "training_code_only",
                "customer_and_prediction": 1,
                "textual_description": 0,
                "training_code": 1,
                "learned_parameters": 0,
                "precomputed_contributions": 0,
                "exact_feature_explanation_possible": 0,
            },
            {
                "condition": "learned_parameters",
                "customer_and_prediction": 1,
                "textual_description": 0,
                "training_code": 0,
                "learned_parameters": 1,
                "precomputed_contributions": 0,
                "exact_feature_explanation_possible": 1,
            },
            {
                "condition": "full_structured_package",
                "customer_and_prediction": 1,
                "textual_description": 0,
                "training_code": 0,
                "learned_parameters": 1,
                "precomputed_contributions": 1,
                "exact_feature_explanation_possible": 1,
            },
        ]
    )
    design["condition_label_visible_to_llm"] = 0
    design["sufficiency_rule_stated_to_llm"] = 0
    design["neutral_task_text"] = (
        EXPLANATION_TASK
        if EXPERIMENT_TASK == "explanation"
        else RECONSTRUCTION_TASK
    ).strip()
    design.to_csv(RESULTS_DIR / "experimental_design.csv", index=False)

    if not RUN_API:
        print(
            "\n[DRY RUN] RUN_API=0. Model, ground truth, and all prompt previews "
            f"were saved under: {RESULTS_DIR.resolve()}"
        )
        print("Set RUN_API=1 to call the LLM.")
        return

    total_calls = len(cases) * len(CONDITIONS) * N_REPETITIONS
    print(
        f"\nRunning {total_calls} API calls: {len(cases)} cases x "
        f"{len(CONDITIONS)} conditions "
        f"x {N_REPETITIONS} repetitions using {MODEL_ID}."
    )

    results = run_experiment(cases)
    results.to_csv(RESULTS_DIR / "api_results_final.csv", index=False)

    summary = summarize_results(results)
    summary.to_csv(RESULTS_DIR / "condition_summary.csv")

    review = create_human_review_template(results)
    review.to_csv(RESULTS_DIR / "human_review_template.csv", index=False)

    print("\nMean metrics by condition:")
    show(summary)
    print(f"\nAll outputs saved under: {RESULTS_DIR.resolve()}")


if __name__ == "__main__":
    main()


Loading South German Credit from UCI...
Dataset shape: X=(1000, 20), y=(1000,)

Training one reproducible frozen logistic-regression model...
{
  "accuracy": 0.7133333333333334,
  "roc_auc": 0.7512169312169312,
  "pr_auc": 0.587028928353325,
  "confusion_matrix": [
    [
      179,
      31
    ],
    [
      55,
      35
    ]
  ],
  "n_train": 700,
  "n_test": 300,
  "n_transformed_features": 54,
  "positive_class": "1 = bad credit risk",
  "threshold": 0.5
}

Selected pilot cases:


,applicant_id,y_true,p_bad_credit,y_pred,outcome
case_id,,,,,
339,SGC0340,0,0.003344,0,TN
158,SGC0159,1,0.959715,1,TP
348,SGC0349,0,0.500463,1,FP



Running 15 API calls: 3 cases x 5 conditions x 1 repetitions using gpt-5.6-sol.
repetition=1 applicant=SGC0340 condition=customer_prediction_only
repetition=1 applicant=SGC0340 condition=textual_description
repetition=1 applicant=SGC0340 condition=training_code_only
repetition=1 applicant=SGC0340 condition=learned_parameters
repetition=1 applicant=SGC0340 condition=full_structured_package
repetition=1 applicant=SGC0159 condition=customer_prediction_only
repetition=1 applicant=SGC0159 condition=textual_description
repetition=1 applicant=SGC0159 condition=training_code_only
repetition=1 applicant=SGC0159 condition=learned_parameters
repetition=1 applicant=SGC0159 condition=full_structured_package
repetition=1 applicant=SGC0349 condition=customer_prediction_only
repetition=1 applicant=SGC0349 condition=textual_description
repetition=1 applicant=SGC0349 condition=training_code_only
repetition=1 applicant=SGC0349 condition=learned_parameters
repetition=1 applicant=SGC0349 condition=full_st

,explanation_possible_correct,appropriate_abstention,missing_model_information_identified,unsupported_feature_claim,logit_abs_error,probability_abs_error,reconstructed_class_error,top5_recall,direction_accuracy,contribution_mae,hallucination_rate,too_many_features,prompt_character_count,prompt_word_count,prompt_tokens,completion_tokens,total_tokens
condition,,,,,,,,,,,,,,,,,
customer_prediction_only,1.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1164.666667,121.0,1044.333333,596.000000,1640.333333
full_structured_package,1.0,1.0,NaN,0.0,NaN,NaN,NaN,1.0,1.0,1.725083e-11,0.0,0.0,12915.333333,787.0,4711.333333,1106.666667,5818.000000
learned_parameters,1.0,1.0,NaN,0.0,NaN,NaN,NaN,1.0,1.0,2.391750e-11,0.0,0.0,6416.333333,665.0,2906.333333,2374.666667,5281.000000
textual_description,1.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,3003.666667,369.0,1383.333333,756.333333,2139.666667
training_code_only,1.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,2505.666667,236.0,1350.333333,649.666667,2000.000000



All outputs saved under: /content/results_sgc_llm2/explanation


In [4]:
from pathlib import Path
from google.colab import files
import shutil

# Path to the experiment results directory
results_dir = Path("/content/results_sgc_llm2")

if not results_dir.exists():
    raise FileNotFoundError(
        f"Results directory was not found: {results_dir}"
    )

# Display all files that will be included
print("Files to be included in the ZIP archive:\n")

for path in sorted(results_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(results_dir))

# Create the ZIP archive
zip_base_path = "/content/results_sgc_llm2"

zip_path = shutil.make_archive(
    base_name=zip_base_path,
    format="zip",
    root_dir=results_dir.parent,
    base_dir=results_dir.name,
)

print(f"\nZIP archive created successfully: {zip_path}")

# Download the ZIP archive
files.download(zip_path)

Files to be included in the ZIP archive:

explanation/api_results_checkpoint.csv
explanation/api_results_final.csv
explanation/compact_llm_answers.jsonl
explanation/condition_summary.csv
explanation/experimental_design.csv
explanation/fitted_coefficients.csv
explanation/frozen_pipeline.joblib
explanation/ground_truth_cases.csv
explanation/ground_truth_contributions.csv
explanation/human_review_template.csv
explanation/model_performance.json
explanation/prompt_previews/SGC0159_customer_prediction_only.txt
explanation/prompt_previews/SGC0159_full_structured_package.txt
explanation/prompt_previews/SGC0159_learned_parameters.txt
explanation/prompt_previews/SGC0159_textual_description.txt
explanation/prompt_previews/SGC0159_training_code_only.txt
explanation/prompt_previews/SGC0340_customer_prediction_only.txt
explanation/prompt_previews/SGC0340_full_structured_package.txt
explanation/prompt_previews/SGC0340_learned_parameters.txt
explanation/prompt_previews/SGC0340_textual_description.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>